# The Elo Derivation: Adapting Chess Elo to Football

Standard Elo (originally designed for chess) is designed for two-player, zero-sum games with binary outcomes (win/loss) or draws. In football, standard Elo is insufficient because:
1. **Margin of Victory:** A 5-0 win indicates a much larger disparity in team strength than a 1-0 win, which standard Elo treats identically.
2. **Home Field Advantage:** Playing at home provides a significant performance boost (~100 Elo points equivalent).
3. **Match Importance:** A World Cup final has significantly higher stakes than an international friendly.

To address these issues, we implement a **modified Elo rating system** based on the official World Football Elo Ratings formula:

$$R_{new} = R_{old} + K \cdot G \cdot (W - W_e)$$

Where:
- $R_{new}$ and $R_{old}$ are the updated and previous Elo ratings.
- $K$ is the match importance factor (e.g., 60 for World Cup, 20 for friendly).
- $G$ is the goal differential multiplier.
- $W$ is the actual result (1.0 for win, 0.5 for draw, 0.0 for loss).
- $W_e$ is the expected win probability.

### 1. Expected Win Probability ($W_e$)

We calculate the expected outcome using the logistic function, incorporating home field advantage ($H = 100$):

$$W_e = \frac{1}{1 + 10^{-\frac{R_{home} + H - R_{away}}{400}}}$$

If the match is played on neutral ground, $H = 0$.

### 2. Goal Differential Multiplier ($G$)

To reward margin of victory, we introduce a multiplier $G$ based on the absolute goal difference $d$:

- If $d \le 1$: $G = 1$
- If $d = 2$: $G = 1.5$
- If $d \ge 3$: $G = \frac{11 + d}{8}$

In [ ]:
import numpy as np
import pandas as pd

def get_goal_diff_multiplier(d):
    if d <= 1:
        return 1.0
    elif d == 2:
        return 1.5
    else:
        return (11.0 + d) / 8.0

# Display multipliers for different goal differences
for gd in range(0, 7):
    print(f"Goal Difference {gd} -> G = {get_goal_diff_multiplier(gd):.3f}")

### 3. Match Importance ($K$-Factor)

We scale the adjustments based on the tournament type:
- **$K=60$**: FIFA World Cup finals
- **$K=50$**: Major continental championships (Euros, Copa América, etc.)
- **$K=40$**: World Cup/continental qualifiers
- **$K=20$**: Friendly matches
- **$K=30$**: Other minor tournaments

### 4. Mathematical Derivation Proof of expected score

The expected score represents the long-term average score of Home vs Away. Since a win is 1, draw is 0.5, and loss is 0:

$$E[W] = 1 \cdot P(Win) + 0.5 \cdot P(Draw) + 0 \cdot P(Loss)$$

Under a standard logistic model, this expected value matches $W_e$ directly, allowing us to compute prediction error $(W - W_e)$ and perform stochastic gradient-style updates to the latent skill ratings (Elo).